Docs:<br>
- [ReportLab Docs](https://docs.reportlab.com/reportlab/userguide/ch1_intro/)
- [StreamLit Gallery - voor ophalen van data](https://streamlit.io/gallery)

<br>

Basisdingen:
[How to iterate over all or certain columns of a df](https://www.geeksforgeeks.org/python/loop-or-iterate-over-all-or-certain-columns-of-a-dataframe-in-python-pandas/) 
<br>
Om naar te kijken: <br>
-  [Google resultaten concepts](https://www.google.com/search?q=app+pc+for+brainstorming+with+drawing+tablet&num=10&sca_esv=9151e0e90600ee3c&sxsrf=ANbL-n7LjWhKl-hEHRYIRnvv6TYRhGIzTA%3A1774340841409&ei=6UrCaenXGMqLi-gPo-_UgAM&biw=1712&bih=1326&ved=0ahUKEwip8L_cjriTAxXKxQIHHaM3FTAQ4dUDCBE&uact=5&oq=app+pc+for+brainstorming+with+drawing+tablet&gs_lp=Egxnd3Mtd2l6LXNlcnAiLGFwcCBwYyBmb3IgYnJhaW5zdG9ybWluZyB3aXRoIGRyYXdpbmcgdGFibGV0MgUQIRigATIFECEYoAEyBRAhGKABSMY-UABY1j1wBngBkAEAmAFwoAGwHaoBBDQ5LjG4AQPIAQD4AQGYAjigArUfwgILEAAYgAQYkQIYigXCAgoQABiABBhDGIoFwgIQEC4YgAQY0QMYQxjHARiKBcICBRAAGIAEwgILEC4YgAQY0QMYxwHCAgUQLhiABMICBhAAGBYYHsICBxAAGIAEGA3CAgYQABgNGB7CAgsQABiABBiGAxiKBcICBRAAGO8FwgIIEAAYgAQYogTCAgcQIRigARgKwgIFECEYnwXCAgQQIRgVmAMAkgcENTQuMqAH3JsCsgcENDguMrgHkh_CBwkwLjI5LjI2LjHIB6EBgAgA&sclient=gws-wiz-serp)
<br>

Aantekeningen:
<br>

Hoe zorg ik dat ik meerdere inputs df's kan verwerken in één uiteindelijke score?

<br>

Handig naslag werk

- [Trapezoidal membership functions](https://www.mathworks.com/help/fuzzy/trapmf.html)



In [38]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import tempfile
from pathlib import Path
import os
import glob
import skfuzzy as fuzz

In [39]:
#Tijdelijke oplossing totdat ik andere manier heb gevonden om input te krijgen
folder_path = r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen'
pattern = os.path.join(folder_path, '*.csv')
csv_files = glob.glob(pattern)


print(csv_files)

[]


In [40]:
#Create function to synthesize assesment input
def synthesize_assessment_input(folder_name='data'):
    #Creëer een pad naar de map waar de csv-bestanden zich bevinden en zoek naar alle csv-bestanden in die map
    base_path = Path.cwd()/folder_name #Path.cwd() geeft de huidige werkmap terug. Base_path is die map plus de mapnaam
    pattern = '*.csv'
    csv_files = list(base_path.glob(pattern))

    if not csv_files: #als er geen csv-bestanden zijn gevonden, geef een foutmelding
        raise FileNotFoundError(f"No CSV files found in the folder: {base_path}")

    all_data = []

    
    #Doorloop alle csv bestanden en voeg ze samen in een dataframe
    for file in csv_files:
        temp_df = pd.read_csv(file, sep=',', encoding='utf-8')
        all_data.append(temp_df)

    #Combineer alle dataframes in één dataframe
    combined_df = pd.concat(all_data, ignore_index=True)

    #Logica om de gecombineerde dataframe te verwerken en te synthetiseren 
    aggregated_logic = {
        'formeel': 'mean',
        'reward_power': 'mean',
        'coercive': 'mean',
        'expert': 'mean',
        'informational': 'mean',
        'pragmatic': 'mean',
        'moral': 'mean',
        'urgency': 'mean',
        'active': 'mean',
        #'tekst_kolom': lambda x: ' | '.join(set(x)) voor mogelijke tekstuele kolommen later

    }

    synthesis = combined_df.groupby('stakeholder').agg(aggregated_logic).reset_index()
    #Mogelijk later toevoegen om af te ronden
    return synthesis

In [41]:
df = synthesize_assessment_input()
print(df.head())

      stakeholder  formeel  reward_power  coercive  expert  informational  \
0   Stakeholder B      2.2           3.4       4.0     3.0            2.8   
1   Stakeholder C      3.2           2.2       2.4     3.4            3.8   
2   Stakeholder E      4.6           5.0       5.0     5.0            5.0   
3  power_dominant      4.8           4.4       4.0     3.4            3.2   
4   stakeholder A      2.4           3.6       3.2     2.6            2.4   

   pragmatic  moral  urgency  active  
0        4.6    2.0      2.6     2.0  
1        2.4    3.0      3.6     2.8  
2        5.0    5.0      3.0     5.0  
3        2.6    2.8      1.6     1.4  
4        3.6    2.8      3.2     2.4  


In [42]:
def calculate_profile_score(df, attribute_prefix):
   return (df[f'{attribute_prefix}_min'] + 2 * df[f'{attribute_prefix}_mean'] + df[f'{attribute_prefix}_max']) / 4
  

def fuzzy_synthesize(folder_name = 'data_2'):
   base_path = Path.cwd()/folder_name #Path.cwd() geeft de huidige werkmap terug. Base_path is die map plus de mapnaam
   pattern = '*.csv'
   csv_files = list(base_path.glob(pattern))

   if not csv_files: #als er geen csv-bestanden zijn gevonden, geef een foutmelding
     raise FileNotFoundError(f"No CSV files found in the folder: {base_path}")
   
   all_data=[pd.read_csv(file, sep=',', encoding='utf-8') for file in csv_files]
   df = pd.concat(all_data, ignore_index=True)

   #Definiëren van de attributen per vraag
   df['power_score'] = df[['vraag_1', 'vraag_2', 'vraag_3']].mean(axis=1)
   df['legitimacy_score'] = df[['vraag_4', 'vraag_5', 'vraag_6']].mean(axis=1)
   df['urgency_score'] = df[['vraag_7', 'vraag_8', 'vraag_9']].mean(axis=1)


   #Aggregeren van de score per attribuut per stakeholder (stap 2 van Poplawska)
   synthesis = df.groupby('stakeholder').agg(
       #power attribuut
       power_mean = ('power_score', 'mean'),
       power_max = ('power_score', 'max'),
       power_min = ('power_score', 'min'),

       #legitimacy attribuut
       legitimacy_mean = ('legitimacy_score', 'mean'),
       legitimacy_max = ('legitimacy_score', 'max'),
       legitimacy_min = ('legitimacy_score', 'min'),

       #urgency attribuut
       urgency_mean = ('urgency_score', 'mean'),
       urgency_max = ('urgency_score', 'max'),
       urgency_min = ('urgency_score', 'min')
   ).reset_index()

   #Berekenen van de salience scores (stap 3 van Poplawska)

   synthesis['salience_mean'] = synthesis[['power_mean', 'legitimacy_mean', 'urgency_mean']].mean(axis=1)
   synthesis['salience_max'] = synthesis[['power_max', 'legitimacy_max', 'urgency_max']].max(axis=1)
   synthesis['salience_min'] = synthesis[['power_min', 'legitimacy_min', 'urgency_min']].min(axis=1)

   synthesis['profile_score_power'] = calculate_profile_score(synthesis, 'power')
   synthesis['profile_score_legitimacy'] = calculate_profile_score(synthesis, 'legitimacy')
   synthesis['profile_score_urgency'] = calculate_profile_score(synthesis, 'urgency')
   synthesis['profile_score_salience'] = calculate_profile_score(synthesis, 'salience')
  
   
   return synthesis

In [43]:
#test van de fuzzy synthese functie
df = fuzzy_synthesize()
print(df.head())

     stakeholder  power_mean  power_max  power_min  legitimacy_mean  \
0  stakeholder A    2.666667   3.000000   2.333333         1.666667   
1  stakeholder B    2.500000   2.666667   2.333333         1.833333   
2  stakeholder C    2.000000   3.000000   1.000000         1.666667   

   legitimacy_max  legitimacy_min  urgency_mean  urgency_max  urgency_min  \
0        2.000000        1.333333      1.833333     2.333333     1.333333   
1        2.333333        1.333333      1.500000     1.666667     1.333333   
2        2.000000        1.333333      1.166667     1.333333     1.000000   

   salience_mean  salience_max  salience_min  profile_score_power  \
0       2.055556      3.000000      1.333333             2.666667   
1       1.944444      2.666667      1.333333             2.500000   
2       1.611111      3.000000      1.000000             2.000000   

   profile_score_legitimacy  profile_score_urgency  profile_score_salience  
0                  1.666667               1.833333  

In [44]:
def equation_1(min_val, mean_val, max_val):
    return (min_val + 2 * mean_val + max_val) / 4

#Fuzzificatie
def trap_mf(x, a, b, c, d):
    if x <= a or x >= d:
        return 0
    elif a < x < b:
        return (x - a) / (b - a)
    elif b <= x <= c:
        return 1
    elif c < x < d:
        return (d - x) / (d - c)
    

def trap_mf_vectorized(x, a, b, c, d):
    """
    Vectorized version of the Trapezoidal Membership Function 
    that works with Pandas Series / Numpy Arrays.
    """
    conditions = [
        (x <= a) | (x >= d),         # Case: Outside boundaries
        (a < x) & (x < b),           # Case: Ramp up
        (b <= x) & (x <= c),         # Case: Flat top (Full membership)
        (c < x) & (x < d)            # Case: Ramp down
    ]
    
    choices = [
        0.0,                         # Outside
        (x - a) / (b - a) if b > a else 1.0,  # Ramp up
        1.0,                         # Flat top
        (d - x) / (d - c) if d > c else 1.0   # Ramp down
    ]
    
    return np.select(conditions, choices, default=0.0)
    

ATTRIBUTE_CONFIG = {
    'power': {
        'Low': [0, 0, 0.6, 1.2],    # Example: boundaries for Power
        'High': [0.6, 1.2, 3, 3]
    },
    'urgency': {
        'Low': [0, 0, 0.6, 1.2],    # Often the same as Power in this paper
        'High': [0.6, 1.2, 3, 3]
    },
    'legitimacy': {
        'Absent': [0, 0, 0, 0], # Legitimacy often uses Absent/Present
        'Present': [0, 0.6, 2.4, 3]
    },
    'salience': {                   # Final Salience has THREE categories
        'Low': [0, 0, 0.6, 1.2],
        'Moderate': [0.6, 1.2, 1.8, 2.4],
        'High': [1.8, 2.4, 3, 3]
    }
}

def get_membership(attribute_name, value):
    config = (ATTRIBUTE_CONFIG[attribute_name])
    results = {}
    for label, params in config.items():
        results[label] = trap_mf_vectorized(value, *params)
    # Implementation for getting membership values based on attribute name and value
    return results


def get_salience_score(df):
    salience_scores = []



    df['power_membership_low'] = get_membership('power', df['profile_score_power'])['Low']
    df['power_membership_high'] = get_membership('power', df['profile_score_power'])['High']
    df['urgency_membership_low'] = get_membership('urgency', df['profile_score_urgency'])['Low']
    df['urgency_membership_high'] = get_membership('urgency', df['profile_score_urgency'])['High']
    df['legitimacy_membership_absent'] = get_membership('legitimacy', df['profile_score_legitimacy'])['Absent']
    df['legitimacy_membership_present'] = get_membership('legitimacy', df['profile_score_legitimacy'])['Present']



   
    return df



In [45]:
#Creëren van Universele sets voor de attributen

x_power = np.arange(0,3.1,0.1)
x_legitimacy = np.arange(0,3.1,0.1)
x_urgency = np.arange(0,3.1,0.1)

#Maken van fuzzy membershipfuncties voor elk attribuut

power_low = fuzz.trapmf(x_power, list(ATTRIBUTE_CONFIG['power']['Low']))
power_high = fuzz.trapmf(x_power, list(ATTRIBUTE_CONFIG['power']['High']))
legitimacy_low = fuzz.trapmf(x_legitimacy, list(ATTRIBUTE_CONFIG['legitimacy']['Absent']))
legitimacy_high = fuzz.trapmf(x_legitimacy, list(ATTRIBUTE_CONFIG['legitimacy']['Present']))
urgency_low = fuzz.trapmf(x_urgency, list(ATTRIBUTE_CONFIG['urgency']['Low']))
urgency_high = fuzz.trapmf(x_urgency, list(ATTRIBUTE_CONFIG['urgency']['High']))

#plotten van de membershipfuncties
fig, (ax0, ax1, ax2) = plt.subplots(nrows=3, figsize=(8, 9))
ax0.plot(x_power, power_low, 'b', linewidth=1.5, label='Low')
ax0.plot(x_power, power_high, 'g', linewidth=1.5, label='High')

ax0.set_title('Power')
ax0.legend()

ax1.plot(x_legitimacy, legitimacy_low, 'b', linewidth=1.5, label='Absent')
ax1.plot(x_legitimacy, legitimacy_high, 'g', linewidth=1.5, label='Present')

ax1.set_title('Legitimacy')
ax1.legend()

ax2.plot(x_urgency, urgency_low, 'b', linewidth=1.5, label='Low')
ax2.plot(x_urgency, urgency_high, 'g', linewidth=1.5, label='High')

ax2.set_title('Urgency')
ax2.legend()

plt.tight_layout()
plt.savefig('output.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
#toepassen van de functies op data
df['power_low']       = df['profile_score_power'].apply(lambda v: fuzz.interp_membership(x_power, power_low, v))
df['power_high']      = df['profile_score_power'].apply(lambda v: fuzz.interp_membership(x_power, power_high, v))
df['legitimacy_absent']  = df['profile_score_legitimacy'].apply(lambda v: fuzz.interp_membership(x_legitimacy, legitimacy_low, v))
df['legitimacy_present'] = df['profile_score_legitimacy'].apply(lambda v: fuzz.interp_membership(x_legitimacy, legitimacy_high, v))
df['urgency_low']     = df['profile_score_urgency'].apply(lambda v: fuzz.interp_membership(x_urgency, urgency_low, v))
df['urgency_high']    = df['profile_score_urgency'].apply(lambda v: fuzz.interp_membership(x_urgency, urgency_high, v))


In [ ]:
#testen van de membership waarden
print(df[['stakeholder', 'profile_score_power', 'power_low', 'power_high', 'legitimacy_absent', 'legitimacy_present', 'urgency_low', 'urgency_high']].head())

     stakeholder  profile_score_power  power_low  power_high  legitimacy_low  \
0  stakeholder A             2.666667        0.0         1.0             0.0   
1  stakeholder B             2.500000        0.0         1.0             0.0   
2  stakeholder C             2.000000        0.0         1.0             0.0   

   legitimacy_high  urgency_low  urgency_high  
0              1.0     0.000000      1.000000  
1              1.0     0.000000      1.000000  
2              1.0     0.055556      0.944444  


In [ ]:
#definiëren van de fuzzy regels

rule_dormant = np.fmin(df['power_high'], df['legitimacy_absent']), df['urgency_low'])
rule_discretionary = np.fmin(df[power_low], df['legitimacy_present'], df['urgency_low'])
rule_demanding = np.fmin(df['power_low'], df['legitimacy_absent'], df['urgency_high'])
rule_dominant = np.fmin(df['power_high'], df['legitimacy_present'], df['urgency_low'])
rule_dangerous = np.fmin(df['power_high'], df['legitimacy_absent'], df['urgency_high'])
rule_dependent = np.fmin(df['power_low'], df['legitimacy_present'], df['urgency_high'])
rule_definitive = np.fmin(df['power_high'], df['legitimacy_present'], df['urgency_high'])
rule_none = np.fmin(df['power_low'], df['legitimacy_absent'], df['urgency_low'])



In [48]:
#Structureren van data:
def data_structure(df):
    columns = df.columns.tolist()
    for col in columns:
        try:
            all_counts = df[col].value_counts()
            return all_counts
        except ValueError:
            print('wrong values')

In [49]:
def get_strategy(power, legitimacy, urgency): #nodig: power en legitimacy score op basis van input)

    average_score = (power + legitimacy + urgency) / 3
    #oude versie 
    #if average_score >= 11: return "Manage closely"
    #if average_score >= 8: return "Keep satisfied"
    #if average_score >= 5: return "Keep informed"
    #return "Monitor only"
    
    if power >= 4 and legitimacy >= 4 and urgency >= 4: return "Manage closely"
    if (power >= 4 or urgency >= 4 or legitimacy >= 4) and (legitimacy < 4 or urgency < 4 or power < 4): return "keep satisfied"
    if (power >= 3 or urgency >= 3 or legitimacy >= 3) and (legitimacy < 3 or urgency < 3 or power < 3): return "keep informed"
    return "Monitor only"


In [50]:
def create_power_legitimacy_urgency_columns(dataFrame):
    power_columns = ['formeel', 'reward_power', 'coercive', 'expert', 'informational']
    legitimacy_columns = ['pragmatic', 'moral']
    urgency_columns = ['urgency', 'active']
    dataFrame['power_scores'] = dataFrame[power_columns].mean(axis=1)
    dataFrame['legitimacy_scores'] = dataFrame[legitimacy_columns].mean(axis=1)
    dataFrame['urgency_scores'] = dataFrame[urgency_columns].mean(axis=1)

In [51]:
def create_matrix_plot(df):
    fig, ax = plt.subplots(figsize=(6,4))
    ax.scatter(df['power_scores'], df['legitimacy_scores'])

    #Kwadranten indelen
    plt.axhline(3, color='black', linewidth=1)
    plt.axvline(3, color='black', linewidth=1)
    plt.xlim(1,5)
    plt.ylim(1,5)

    plt.xlabel('power (1-5)')
    plt.ylabel('legitimacy (1-5)')
    plt.title('Stakeholder map')

    for i, txt in enumerate(df['stakeholder']):
        ax.annotate(txt, (df['power_scores'].iat[i], df['legitimacy_scores'].iat[i])) #.iat werkt als iloc, maar dan voor specifieke cellen, niet hele rijen of kolommen

    plt.tight_layout()
    plt.show()
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path

In [52]:
#Functie voor het creëren van een venn-diagram van stakeholders.
# We creëren een plot met een driehoek, waarbij elke punt van een driehoek de 'kern' van elk onderdeel van de stakeholder analysere vertegenwoordigd.
# Dit plotten we in een raster.

# De driehoek heeft drie hoeken met ieder dezelfde afstand tot het midden (360/3=120 graden)
#Hoek A (top/power) = 90
#hoek B (linksonder/legitimacy) = 90 + 120 = 210
#hoek C (rechtsonder/urgency) = 90 + 240 = 330
#om de coördinaten te krijgen gebruiken we de volgende formules:
# x = r.cos(theta), waarbij r de afstand tot het midden is en theta de hoek in radialen
# y = r.sin(theta)
# p = power score, l = legitimacy score, u = urgency score, r=1

anchors = {
        'p': {'coord':np.array([np.cos(np.radians(90)), np.sin(np.radians(90))]), 'color':'red', 'label':'Power'},
        'l': {'coord':np.array([np.cos(np.radians(210)), np.sin(np.radians(210))]), 'color':'blue', 'label':'Legitimacy'},
        'u': {'coord':np.array([np.cos(np.radians(330)), np.sin(np.radians(330))]), 'color':'green', 'label':'Urgency'}
        }

print(anchors)

test_p = 4 ** 3
test_l = 2 ** 3
test_u = 4 ** 3

x = (test_p * anchors['p']['coord'][0] + test_l * anchors['l']['coord'][0] + test_u * anchors['u']['coord'][0]) / (test_p + test_l + test_u)
y = (test_p * anchors['p']['coord'][1] + test_l * anchors['l']['coord'][1] + test_u * anchors['u']['coord'][1]) / (test_p + test_l + test_u)

print(x, y)

def calculate_coordinates(s):

    s['total_score'] = s['power_scores'] + s['legitimacy_scores'] + s['urgency_scores']
    x = (s['power_scores'] * anchors['p']['coord'][0] + s['legitimacy_scores'] * anchors['l']['coord'][0] + s['urgency_scores'] * anchors['u']['coord'][0]) / s['total_score']
    y = (s['power_scores'] * anchors['p']['coord'][1] + s['legitimacy_scores'] * anchors['l']['coord'][1] + s['urgency_scores'] * anchors['u']['coord'][1]) / s['total_score']
    return x, y

def calculate_coordinates_2(s, exponent=3): # Use 2 or 3 for spread
    # Amplify the scores to push them away from the center
    p_pull = s['power_scores'] ** exponent
    l_pull = s['legitimacy_scores'] ** exponent
    u_pull = s['urgency_scores'] ** exponent
    
    total_pull = p_pull + l_pull + u_pull
    
    if total_pull == 0: return 0, 0
    
    # Calculate weighted position using the amplified "pull"
    x = (p_pull * anchors['p']['coord'][0] + 
         l_pull * anchors['l']['coord'][0] + 
         u_pull * anchors['u']['coord'][0]) / total_pull
         
    y = (p_pull * anchors['p']['coord'][1] + 
         l_pull * anchors['l']['coord'][1] + 
         u_pull * anchors['u']['coord'][1]) / total_pull
         
    return x, y

def create_venn_diagram(df):
    #creating the triangle coordinates
    anchors = {
        'p': {'coord':np.array([np.cos(np.radians(90)), np.sin(np.radians(90))]), 'color':'red', 'label':'Power'},
        'l': {'coord':np.array([np.cos(np.radians(210)), np.sin(np.radians(210))]), 'color':'blue', 'label':'Legitimacy'},
        'u': {'coord':np.array([np.cos(np.radians(330)), np.sin(np.radians(330))]), 'color':'green', 'label':'Urgency'}
        }

    #creëren van de figuur
    fig, ax = plt.subplots(figsize=(13,13))
    #creëren van de cirkels voor de venn-diagram
    for key, data in anchors.items():
        circle = plt.Circle(data['coord'], 1, color=data['color'], alpha=0.5, label=data['label'])
        ax.add_patch(circle)
        #creëren van lijn tussen de cirkels
        circle_outline = plt.Circle(data['coord'], 1, color=data['color'], fill=False, linewidth=2, alpha=0.3)
        ax.add_patch(circle_outline)


    for index, s in df.iterrows():
        x, y = calculate_coordinates_2(s)
        saliency = (s['power_scores'] + s['legitimacy_scores'] + s['urgency_scores']) / 15 #max score is 15, dus delen door 15 om tussen 0 en 1 te krijgen
        ax.scatter(x,y, alpha=0.8, s=saliency*1000, edgecolors='black', zorder=5)
        ax.text(x, y, s['stakeholder'], fontsize=9, ha='center', va='center', zorder=6)

    # Label the vertices
    plt.text(0, 1.1, "POWER", fontsize=12, ha='center', color='red', fontweight='bold')
    plt.text(-0.9, -0.6, "LEGITIMACY", fontsize=12, ha='center', color='blue', fontweight='bold')
    plt.text(0.9, -0.6, "URGENCY", fontsize=12, ha='center', color='green', fontweight='bold')

    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal') # CRITICAL: Keeps circles circular, not ovals
    plt.axis('off')
    plt.title("Stakeholder Venn-Gravity Map", fontsize=14, pad=20)

    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path





{'p': {'coord': array([6.123234e-17, 1.000000e+00]), 'color': 'red', 'label': 'Power'}, 'l': {'coord': array([-0.8660254, -0.5      ]), 'color': 'blue', 'label': 'Legitimacy'}, 'u': {'coord': array([ 0.8660254, -0.5      ]), 'color': 'green', 'label': 'Urgency'}}
0.3565986956759452 0.20588235294117627


In [ ]:
#Creëren van een 'force field' of 'vector map'
#Dit moet ervoor zorgen dat stakeholders met hoge scores op p, l en u meer in het midden liggen, terwijl lage scores zorgen dat stakeholders buiten de rand komen.

def create_vector_coordinates(s):
    anchors = {
        'p': {'coord':np.array([np.cos(np.radians(90)), np.sin(np.radians(90))])},
        'l': {'coord':np.array([np.cos(np.radians(210)), np.sin(np.radians(210))])},
        'u': {'coord':np.array([np.cos(np.radians(330)), np.sin(np.radians(330))])}
        }
    
    p, l, u = s['power_scores'], s['legitimacy_scores'], s['urgency_scores']
    total = p + l + u
    distance_max = 15
    distance_min = 3

    net_x = (p * anchors['p']['coord'][0] + l * anchors['l']['coord'][0] + u * anchors['u']['coord'][0]) / total
    net_y = (p * anchors['p']['coord'][1] + l * anchors['l']['coord'][1] + u * anchors['u']['coord'][1]) / total

    length = np.sqrt(net_x**2 + net_y**2)

    distance = (distance_max - total) / (distance_max - distance_min) 

    final_x = net_x/length * distance
    final_y = net_y/length * distance

    return final_x, final_y

    #creeëren van de driehoek coördinaten
    
    
    
def create_vector_plot(df):
    #creëren van de figuur
    fig, ax = plt.subplots(figsize=(13,13))
    #creëren van de cirkels voor de venn-diagram
    circles = {'center': {'radius': 0.33, 'text':'most salient', 'text_pos': (0, 0.30)}, 
               'r1': {'radius': 0.66, 'text': 'less salient', 'text_pos': (0, 0.63)}, 
               'r2': {'radius': 1.0, 'text': 'least salient', 'text_pos': (0, 0.93)}}
    for key, data in circles.items():
        circle = plt.Circle((0, 0), data['radius'], color='grey', fill=False, linestyle='dashed', alpha=0.2)
        ax.add_patch(circle)
        ax.text(data['text_pos'][0], data['text_pos'][1], data['text'], fontsize=12, ha='center', va='center', zorder=4)

    for i, s in df.iterrows():
        x, y = create_vector_coordinates(s)
        total = s['power_scores'] + s['legitimacy_scores'] + s['urgency_scores']

        #Teken van de vectoren, waarbij de lengte van de vector afhankelijk is van hoe laag de total score is (hoe lager, hoe langer de vector)

        if total < 15:
            ax.quiver(x, y, -x*0.5, -y*0.5, angles='xy', scale_units='xy', scale=1, 
                  color='gray', alpha=0.3, width=0.003)
            color = plt.cm.RdYlBu_r(total/15) #Kleur afhankelijk van total score, waarbij 0 rood is en 15 blauw

        ax.scatter(x,y,s=(total/15)*1000, color=color, edgecolors='black', zorder=5)
        ax.text(x,y + 0.07, s['stakeholder'], fontsize=9, ha='center', va='center', zorder=6)

    
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    plt.axis('off')
    plt.title("Radial Force Field: Importance = Proximity to Center", pad=20)
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path




    

    
    
    



In [ ]:
#Creëren van 3d scatterplot met plt
def create_3d_plot(df):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    x = df['power_scores']
    y = df['legitimacy_scores']
    z = df['urgency_scores']
    total = x + y + z

    sc = ax.scatter3D(x, y, z, label=df['stakeholder'], c=total, cmap='RdYlBu_r', vmin=3, vmax=15, s=(total/15)*1000, marker = '^')

    for i, txt in enumerate(df['stakeholder']):
        ax.text(x[i], y[i], z[i] + 0.2, txt, size=9, zorder=1, color='k')

    # Proper Colorbar setup
    cbar = plt.colorbar(sc, ax=ax, shrink=0.5, aspect=10)
    cbar.set_label('Salience (Total Score)')

    # Axis labels and title
    ax.set_xlabel('Power')
    ax.set_ylabel('Legitimacy')
    ax.set_zlabel('Urgency')
    plt.title('3D Stakeholder Map')
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path


In [ ]:
create_power_legitimacy_urgency_columns(df)
df['strategy'] = df.apply(lambda row: get_strategy(row['power_scores'], row['legitimacy_scores'], row['urgency_scores']), axis=1)


for i, row in df.iterrows():
    print(f"Stakeholder: {row['stakeholder']}, power: {row['power_scores']}, legitimacy: {row['legitimacy_scores']}, urgency: {row['urgency_scores']}, Strategy: {row['strategy']}")


KeyError: "None of [Index(['formeel', 'reward_power', 'coercive', 'expert', 'informational'], dtype='object')] are in the [columns]"

In [ ]:
#Plot stakeholders op kaart
create_matrix_plot(df)


File location: C:\Users\hans_\AppData\Local\Temp\tmp42m800ue.png


C:\Users\hans_\AppData\Local\Temp\ipykernel_13224\2491134796.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmp42m800ue.png'

In [ ]:
create_power_legitimacy_urgency_columns(df)
create_venn_diagram(df)

File location: C:\Users\hans_\AppData\Local\Temp\tmpc15lykpk.png


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmpc15lykpk.png'

In [ ]:
create_vector_plot(df)

File location: C:\Users\hans_\AppData\Local\Temp\tmpkmieog00.png


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmpkmieog00.png'

In [ ]:
create_3d_plot(df)

File location: C:\Users\hans_\AppData\Local\Temp\tmpdi4mcyg1.png


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmpdi4mcyg1.png'